In [83]:
!pip install optuna

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import TimeSeriesSplit, KFold
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import lightgbm as lgb
import optuna

In [84]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [85]:
df = pd.read_csv('/content/drive/MyDrive/train.csv')

In [86]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 206150 entries, 0 to 206149
Data columns (total 19 columns):
 #   Column                                     Non-Null Count   Dtype  
---  ------                                     --------------   -----  
 0   new_id                                     206150 non-null  int64  
 1   Месяц                                      206150 non-null  int64  
 2   Дата открытия, категориальный              206150 non-null  object 
 3   Торговая площадь, категориальный           206150 non-null  object 
 4   Населенный пункт                           206150 non-null  object 
 5   Регион                                     206150 non-null  object 
 6   Численность населения                      206150 non-null  int64  
 7   Количество домохозяйств                    206150 non-null  int64  
 8   Трафик пеший, в час                        206150 non-null  int64  
 9   Трафик авто, в час                         206150 non-null  int64  
 10  Маркетпл

In [87]:
store_counts = df.groupby('new_id')['Месяц'].nunique()
print(f"Магазинов с 10 месяцами: {(store_counts == 10).sum()} из {len(store_counts)}")

Магазинов с 10 месяцами: 20615 из 20615


In [88]:
df[df['Флаг алкогольной лицензии'] == 0]

,new_id,Месяц,"Дата открытия, категориальный","Торговая площадь, категориальный",Населенный пункт,Регион,Численность населения,Количество домохозяйств,"Трафик пеший, в час","Трафик авто, в час","Маркетплейсы, доставки, постаматы (100 м)",Медицинские уч. и аптеки (300 м),Школы (300 м),Остановки (300 м),Продуктовые магазины (500 м),Пятерочки (500 м),Количество касс,Флаг алкогольной лицензии,РТО
100,10,1,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,27668007.80
101,10,2,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,28691671.27
102,10,3,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,33355460.71
103,10,4,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,30820672.66
104,10,5,Средний по возрасту,Средний,Пушкино г,Московская обл,108191,6830,191,212,7,0,0,1,8,2,5,0,32521965.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206025,21312,6,Новый,Средний,Ижевск г,Удмуртская Респ,618128,6866,113,206,4,5,0,7,7,1,8,0,12010430.99
206026,21312,7,Новый,Средний,Ижевск г,Удмуртская Респ,618128,6866,113,206,4,5,0,7,7,1,8,0,13697087.72
206027,21312,8,Новый,Средний,Ижевск г,Удмуртская Респ,618128,6866,113,206,4,5,0,7,7,1,8,0,15089772.38
206028,21312,9,Новый,Средний,Ижевск г,Удмуртская Респ,618128,6866,113,206,4,5,0,7,7,1,8,0,16337214.10


In [89]:
# ---------- 1. Расширенный feature engineering ----------
df = df.sort_values(['new_id', 'Месяц']).copy()

# Лаги (1,2,3) – уже были
for lag in [1,2,3]:
    df[f'RTO_lag_{lag}'] = df.groupby('new_id')['РТО'].shift(lag)

# Дополнительные скользящие статистики
df['RTO_ma_2'] = df.groupby('new_id')['РТО'].transform(lambda x: x.shift(1).rolling(2).mean())
df['RTO_ma_3'] = df.groupby('new_id')['РТО'].transform(lambda x: x.shift(1).rolling(3).mean())
df['RTO_ma_6'] = df.groupby('new_id')['РТО'].transform(lambda x: x.shift(1).rolling(6).mean())  # полгода

# Относительные изменения
df['RTO_growth_1'] = df.groupby('new_id')['РТО'].pct_change(1)
df['RTO_growth_2'] = df.groupby('new_id')['РТО'].pct_change(2)

# Признаки самого магазина (агрегаты за прошлые месяцы)
store_stats = df[df['Месяц'] <= 9].groupby('new_id')['РТО'].agg(['mean', 'median', 'std', 'min', 'max'])
store_stats.columns = ['store_mean_rto', 'store_median_rto', 'store_std_rto', 'store_min_rto', 'store_max_rto']
df = df.merge(store_stats, on='new_id', how='left')

# Отношение текущего РТО к среднему по магазину (для обучающей выборки – сдвиг, чтобы избежать лика)
df['rto_to_store_mean'] = df['РТО'] / (df['store_mean_rto'] + 1e-6)

# Взаимодействия числовых признаков
df['трафик_на_кассу'] = df['Трафик пеший, в час'] / (df['Количество касс'] + 1)
df['население_на_кассу'] = df['Численность населения'] / (df['Количество касс'] + 1)
df['плотность_пятерочек'] = df['Пятерочки (500 м)'] / (df['Продуктовые магазины (500 м)'] + 1)

# Циклические признаки месяца (синус/косинус)
df['month_sin'] = np.sin(2 * np.pi * df['Месяц'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['Месяц'] / 12)

In [90]:
categorical_cols = ['Дата открытия, категориальный', 'Торговая площадь, категориальный',
                    'Населенный пункт', 'Регион', 'Флаг алкогольной лицензии']

def smooth_target_encoding(train_df, cat_col, target_col, prior_weight=20):
    prior = train_df[target_col].mean()
    agg = train_df.groupby(cat_col)[target_col].agg(['mean', 'count'])
    smoothed = (agg['mean'] * agg['count'] + prior * prior_weight) / (agg['count'] + prior_weight)
    return train_df[cat_col].map(smoothed).fillna(prior)

train_part = df[df['Месяц'] <= 9].copy()
global_prior = train_part['РТО'].mean()

for col in categorical_cols:
    encoded_col = f'{col}_te'
    enc_vals = smooth_target_encoding(train_part, col, 'РТО', prior_weight=20)
    train_part[encoded_col] = enc_vals
    mapping = train_part.set_index(col)[encoded_col].to_dict()
    df[encoded_col] = df[col].map(mapping).fillna(global_prior)

In [91]:
numeric_features = ['Численность населения', 'Количество домохозяйств', 'Трафик пеший, в час',
                    'Трафик авто, в час', 'Маркетплейсы, доставки, постаматы (100 м)',
                    'Медицинские уч. и аптеки (300 м)', 'Школы (300 м)', 'Остановки (300 м)',
                    'Продуктовые магазины (500 м)', 'Пятерочки (500 м)', 'Количество касс',
                    'RTO_lag_1', 'RTO_lag_2', 'RTO_lag_3', 'RTO_ma_2', 'RTO_ma_3', 'RTO_ma_6',
                    'RTO_growth_1', 'RTO_growth_2', 'store_mean_rto', 'store_median_rto', 'store_std_rto',
                    'store_min_rto', 'store_max_rto', 'rto_to_store_mean', 'трафик_на_кассу',
                    'население_на_кассу', 'плотность_пятерочек', 'month_sin', 'month_cos']

# Добавляем target encoded колонки
te_cols = [f'{col}_te' for col in categorical_cols]
feature_cols = numeric_features + te_cols + ['Месяц']

# Удаляем строки с NaN (первые несколько месяцев не имеют лагов)
df_clean = df.dropna(subset=['RTO_lag_1', 'RTO_lag_2', 'RTO_lag_3']).copy()
print(f"После удаления NaN: {df_clean.shape}")

После удаления NaN: (144305, 43)


In [92]:
train = df_clean[df_clean['Месяц'] <= 8].copy()
val = df_clean[df_clean['Месяц'] == 9].copy()
future_features = df_clean[df_clean['Месяц'] == 10].copy()  # используем для получения лагов при прогнозе 11

print(f"Train: {train.shape}, Val: {val.shape}, Future features: {future_features.shape}")

X_train = train[feature_cols]
y_train = np.log1p(train['РТО'])

X_val = val[feature_cols]
y_val = np.log1p(val['РТО'])

Train: (103075, 43), Val: (20615, 43), Future features: (20615, 43)


In [93]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2.0),
        'eval_metric': 'mape',
        'random_state': 42
    }
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train, verbose=False)
    pred = np.expm1(model.predict(X_val))
    mape = mean_absolute_percentage_error(val['РТО'], pred)
    return mape

# Целевая функция для LightGBM
def objective_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 1500),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 50),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 2.0),
        'random_state': 42,
        'verbosity': -1
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    pred = np.expm1(model.predict(X_val))
    mape = mean_absolute_percentage_error(val['РТО'], pred)
    return mape

In [94]:
# Очистка имен колонок от спецсимволов
def clean_column_names(df):
    # Заменяем все символы, кроме букв, цифр и подчеркивания, на подчеркивание
    df.columns = df.columns.str.replace(r'[^\w]', '_', regex=True)
    return df

# Применяем к обучающим и тестовым наборам
X_train = clean_column_names(X_train)
X_val = clean_column_names(X_val)

In [95]:
study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)

[I 2026-05-06 21:11:34,255] A new study created in memory with name: no-name-87e6a0f9-e4ce-4634-8156-8d17d676b4c5


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-05-06 21:12:11,189] Trial 0 finished with value: 0.00883498630023329 and parameters: {'n_estimators': 697, 'max_depth': 10, 'learning_rate': 0.060557309681542874, 'subsample': 0.8895185567308366, 'colsample_bytree': 0.6968325690790614, 'min_child_weight': 4, 'reg_alpha': 0.5182108555887222, 'reg_lambda': 0.01860149151031565}. Best is trial 0 with value: 0.00883498630023329.
[I 2026-05-06 21:13:02,702] Trial 1 finished with value: 0.008540554498964957 and parameters: {'n_estimators': 1185, 'max_depth': 11, 'learning_rate': 0.028870604033503448, 'subsample': 0.8275970737376873, 'colsample_bytree': 0.7603030174384217, 'min_child_weight': 3, 'reg_alpha': 1.737169403512705, 'reg_lambda': 1.2284742364699412}. Best is trial 1 with value: 0.008540554498964957.
[I 2026-05-06 21:13:55,063] Trial 2 finished with value: 0.008519899108286572 and parameters: {'n_estimators': 1172, 'max_depth': 9, 'learning_rate': 0.023721345132919167, 'subsample': 0.9083978002389206, 'colsample_bytree': 0.83

In [96]:
study_lgb = optuna.create_study(direction='minimize')
study_lgb.optimize(objective_lgb, n_trials=30, show_progress_bar=True)

[I 2026-05-06 21:36:21,438] A new study created in memory with name: no-name-00f74235-852d-48c9-9186-cf303eab23a5


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-05-06 21:36:42,500] Trial 0 finished with value: 0.009435053738934636 and parameters: {'n_estimators': 960, 'max_depth': 6, 'learning_rate': 0.026616038794262175, 'subsample': 0.6915938427513411, 'colsample_bytree': 0.613056015821179, 'num_leaves': 130, 'min_child_samples': 7, 'reg_alpha': 1.7697741623332504, 'reg_lambda': 1.7479524188186681}. Best is trial 0 with value: 0.009435053738934636.
[I 2026-05-06 21:37:11,642] Trial 1 finished with value: 0.010131605620040323 and parameters: {'n_estimators': 1402, 'max_depth': 6, 'learning_rate': 0.04833232416676535, 'subsample': 0.6116957380591052, 'colsample_bytree': 0.74392796457937, 'num_leaves': 235, 'min_child_samples': 15, 'reg_alpha': 0.878736189369536, 'reg_lambda': 0.23313430375039967}. Best is trial 0 with value: 0.009435053738934636.
[I 2026-05-06 21:37:27,699] Trial 2 finished with value: 0.009622714576738158 and parameters: {'n_estimators': 536, 'max_depth': 12, 'learning_rate': 0.02126687806938381, 'subsample': 0.986902

In [97]:
print(f"Лучший MAPE XGB: {study_xgb.best_value:.4f}")
print("\nЛучшие параметры XGBoost:", study_xgb.best_params)
print(f"Лучший MAPE LightGBM: {study_lgb.best_value:.4f}")
print(f"Лучшие параметры: {study_lgb.best_params}")

Лучший MAPE XGB: 0.0078

Лучшие параметры XGBoost: {'n_estimators': 1498, 'max_depth': 9, 'learning_rate': 0.017460970949940667, 'subsample': 0.889827350439843, 'colsample_bytree': 0.7340184731512398, 'min_child_weight': 1, 'reg_alpha': 0.29592556798044845, 'reg_lambda': 1.860689819679372}
Лучший MAPE LightGBM: 0.0088
Лучшие параметры: {'n_estimators': 1491, 'max_depth': 8, 'learning_rate': 0.012806980150827638, 'subsample': 0.7261997134970036, 'colsample_bytree': 0.7785231252340741, 'num_leaves': 77, 'min_child_samples': 5, 'reg_alpha': 0.7379353801592768, 'reg_lambda': 0.4069046940742097}


In [98]:
train_full = df_clean[df_clean['Месяц'] <= 9].copy()
X_full = train_full[feature_cols]
X_full = clean_column_names(X_full)
y_full = np.log1p(train_full['РТО'])

# XGBoost с лучшими параметрами
xgb_params = study_xgb.best_params
xgb_model = xgb.XGBRegressor(**xgb_params, random_state=42)
xgb_model.fit(X_full, y_full)

# LightGBM с лучшими параметрами
lgb_params = study_lgb.best_params
lgb_model = lgb.LGBMRegressor(**lgb_params, random_state=42, verbosity=-1)
lgb_model.fit(X_full, y_full)

LGBMRegressor(colsample_bytree=0.7785231252340741,
              learning_rate=0.012806980150827638, max_depth=8,
              min_child_samples=5, n_estimators=1491, num_leaves=77,
              random_state=42, reg_alpha=0.7379353801592768,
              reg_lambda=0.4069046940742097, subsample=0.7261997134970036,
              verbosity=-1)

In [99]:
# Берём данные месяца 10
month10 = df_clean[df_clean['Месяц'] == 10].copy()
# Для каждого магазина подтянем РТО месяца 9 и 8 из исходного df (не обрезанного)
# Проще: создадим словарь с лагами
store_rto = df[['new_id', 'Месяц', 'РТО']].drop_duplicates().set_index(['new_id', 'Месяц'])['РТО']

def get_lag(row, lag):
    month = row['Месяц'] + lag  # месяц 10 + 1 = 11? нет, нам нужно для месяца 11: лаг1 = РТО месяца 10
    # Для строки месяца 11, лаг1 = РТО_10, лаг2 = РТО_9, лаг3 = РТО_8
    target_month = 11 - lag
    return store_rto.get((row['new_id'], target_month), np.nan)

month11 = month10.copy()
month11['Месяц'] = 11
month11['RTO_lag_1'] = month10.apply(lambda r: get_lag(r, 1), axis=1)   # РТО_10
month11['RTO_lag_2'] = month10.apply(lambda r: get_lag(r, 2), axis=1)   # РТО_9
month11['RTO_lag_3'] = month10.apply(lambda r: get_lag(r, 3), axis=1)   # РТО_8
# Пересчитаем скользящие на основе новых лагов (можно упрощённо)
month11['RTO_ma_2'] = (month11['RTO_lag_1'] + month11['RTO_lag_2']) / 2
month11['RTO_ma_3'] = (month11['RTO_lag_1'] + month11['RTO_lag_2'] + month11['RTO_lag_3']) / 3
month11['RTO_ma_6'] = month11['RTO_ma_3']  # упрощённо, для полноты
month11['RTO_growth_1'] = month11['RTO_lag_1'] / month11['RTO_lag_2'] - 1
month11['RTO_growth_2'] = month11['RTO_lag_1'] / month11['RTO_lag_3'] - 1
month11['rto_to_store_mean'] = month11['RTO_lag_1'] / (month11['store_mean_rto'] + 1e-6)

# Остальные признаки остаются как в month10
X_month11 = month11[feature_cols]
X_month11 = clean_column_names(X_month11)

In [100]:
# Предсказание ансамблем
pred_xgb_log = xgb_model.predict(X_month11)
pred_lgb_log = lgb_model.predict(X_month11)

# Среднее арифметическое в логарифмическом пространстве, затем экспонента
pred_ensemble_log = (pred_xgb_log + pred_lgb_log) / 2
pred_final = np.expm1(pred_ensemble_log)

In [101]:
xgb_val = xgb.XGBRegressor(**study_xgb.best_params, random_state=42)
xgb_val.fit(X_train, y_train)
pred_xgb_val = np.expm1(xgb_val.predict(X_val))

lgb_val = lgb.LGBMRegressor(**study_lgb.best_params, random_state=42, verbosity=-1)
lgb_val.fit(X_train, y_train)
pred_lgb_val = np.expm1(lgb_val.predict(X_val))

pred_val_ensemble = (pred_xgb_val + pred_lgb_val) / 2
mape_val = mean_absolute_percentage_error(val['РТО'], pred_val_ensemble)

print(f"\n=== Валидация на месяце 9 ===")
print(f"MAPE XGBoost: {mean_absolute_percentage_error(val['РТО'], pred_xgb_val):.4f}")
print(f"MAPE LightGBM: {mean_absolute_percentage_error(val['РТО'], pred_lgb_val):.4f}")
print(f"MAPE ансамбля: {mape_val:.4f}")


=== Валидация на месяце 9 ===
MAPE XGBoost: 0.0078
MAPE LightGBM: 0.0088
MAPE ансамбля: 0.0077


In [102]:
submission = pd.DataFrame({
    'new_id': month11['new_id'].values,
    'rto': pred_final
})

# Убедимся, что new_id уникальны
submission = submission.drop_duplicates(subset=['new_id'])
print(f"\nСформировано {len(submission)} предсказаний для месяца 11")

# Сохраняем
submission.to_csv('prediction_month11.csv', index=False)
print("Файл prediction_month11.csv сохранён")


Сформировано 20615 предсказаний для месяца 11
Файл prediction_month11.csv сохранён


In [103]:
importance = pd.DataFrame({
    'feature': X_full.columns,
    'importance_gain': xgb_model.feature_importances_
}).sort_values('importance_gain', ascending=False)
print("\nТоп-15 важных признаков (XGBoost):")
print(importance.head(15))


Топ-15 важных признаков (XGBoost):
                  feature  importance_gain
11              RTO_lag_1         0.477934
19         store_mean_rto         0.380526
20       store_median_rto         0.054980
23          store_max_rto         0.037339
14               RTO_ma_2         0.023006
24      rto_to_store_mean         0.010191
17           RTO_growth_1         0.006297
18           RTO_growth_2         0.003825
22          store_min_rto         0.001342
15               RTO_ma_3         0.000851
12              RTO_lag_2         0.000676
21          store_std_rto         0.000516
29              month_cos         0.000345
0   Численность_населения         0.000301
28              month_sin         0.000261
